# Exact symWeighted ring/dual factorisation — fail-closed Lean verification

This notebook checks one immutable WIP SHA. It reruns the preregistered exact symWeighted gate in normal and optimized Python before building the Lean module, the full core, and the permanent oracle. It licenses only the exact entrywise ring/dual factorisation theorem under `0 < β` and `0 ≤ γ`; it does not certify a Jordan–Wigner transform, an operator-norm estimate, either sector obligation, or the uniform spatial-ring bound.

In [ ]:
from pathlib import Path
import datetime, hashlib, json, os, platform, re, shutil, subprocess, sys, tempfile

REPO_URL = 'https://github.com/lluiseriksson/THE-ERIKSSON-PROGRAMME.git'
EXPECTED_SHA = '89c19ab2b328061b74d194dc75ebfdbba04610c6'
EXPECTED_TOOLCHAIN = 'leanprover/lean4:v4.29.0-rc6'
EXPECTED_LEAN_COMMIT = '00659f8e6071d7e46131ed643bf8003b99b044e9'
EXPECTED_MATHLIB = '07642720480157414db592fa85b626dafb71355b'
EXPECTED_GATE_SHA256 = 'a95e66da0ee527b1776ceb3d13d83760d1fd88cc9227ebea668a2b98ca1946cf'
EXPECTED_GATE_COUNT = 5460
EXPECTED_MODULE_JOBS = 8172
EXPECTED_CORE_JOBS = 8466
RUN_ROOT = Path(tempfile.mkdtemp(prefix='spatial-symweighted-lean-'))
REPO = RUN_ROOT / 'repo'
ARTIFACTS = RUN_ROOT / 'artifacts'
ARTIFACTS.mkdir()
TRANSCRIPT = ARTIFACTS / 'transcript.txt'

def log(value):
    value = str(value)
    print(value, flush=True)
    with TRANSCRIPT.open('a', encoding='utf-8', newline='\n') as stream:
        stream.write(value + '\n')

def run(command, cwd=None, env=None, allow_failure=False):
    shown = ' '.join(map(str, command))
    log(f'$ {shown}')
    started = datetime.datetime.now(datetime.timezone.utc)
    result = subprocess.run(command, cwd=cwd, env=env, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    elapsed = (datetime.datetime.now(datetime.timezone.utc) - started).total_seconds()
    log(result.stdout.rstrip())
    log(f'[exit {result.returncode}; wall_seconds={elapsed}]')
    if result.returncode and not allow_failure:
        raise RuntimeError(f'command failed ({result.returncode}): {shown}')
    return result

log('SPATIAL SYMWEIGHTED LEAN COLAB RUN')
log(f'utc_start={datetime.datetime.now(datetime.timezone.utc).isoformat()}')
log(f'platform={platform.platform()}')
log(f'python={platform.python_version()}')
log(f'cpu_count={os.cpu_count()}')
log(Path('/proc/cpuinfo').read_text(errors='replace').splitlines()[4] if Path('/proc/cpuinfo').exists() else 'cpuinfo=unavailable')
log(Path('/proc/meminfo').read_text(errors='replace').splitlines()[0] if Path('/proc/meminfo').exists() else 'meminfo=unavailable')
run(['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO)])
run(['git', 'checkout', '--detach', EXPECTED_SHA], cwd=REPO)
actual_sha = run(['git', 'rev-parse', 'HEAD'], cwd=REPO).stdout.strip()
if actual_sha != EXPECTED_SHA:
    raise RuntimeError(f'SHA mismatch: {actual_sha} != {EXPECTED_SHA}')
toolchain = (REPO / 'lean-toolchain').read_text(encoding='utf-8').strip()
if toolchain != EXPECTED_TOOLCHAIN:
    raise RuntimeError(f'toolchain mismatch: {toolchain} != {EXPECTED_TOOLCHAIN}')
manifest = json.loads((REPO / 'lake-manifest.json').read_text(encoding='utf-8'))
entries = [package for package in manifest['packages'] if package.get('name') == 'mathlib']
if len(entries) != 1 or entries[0].get('rev') != EXPECTED_MATHLIB:
    raise RuntimeError(f'mathlib pin mismatch: {entries}')
gate = REPO / 'scripts' / 'judge_spatial_symweighted_factorization.py'
gate_hash = hashlib.sha256(gate.read_bytes()).hexdigest()
if gate_hash != EXPECTED_GATE_SHA256:
    raise RuntimeError(f'gate SHA-256 mismatch: {gate_hash}')
for flags in ([], ['-O']):
    gate_run = run([sys.executable, *flags, str(gate)], cwd=REPO)
    lines = gate_run.stdout.splitlines()
    if len(lines) != 1:
        raise RuntimeError(f'gate emitted stale or extra output under flags {flags}')
    payload = json.loads(lines[0])
    fields = ('configuration_pairs_checked', 'scale_mutations_rejected',
        'source_closing_bond_mutations_rejected', 'target_closing_bond_mutations_rejected')
    if payload.get('status') != 'PASS' or any(payload.get(k) != EXPECTED_GATE_COUNT for k in fields):
        raise RuntimeError(f'gate PASS/count mismatch under flags {flags}: {payload}')
    if payload.get('ring_sizes') != [1, 2, 3, 4, 5, 6]:
        raise RuntimeError(f'gate ring-size mismatch under flags {flags}: {payload}')
log(f'repo_sha={actual_sha}')
log(f'lean_toolchain={toolchain}')
log(f'mathlib_pin={entries[0].get("rev")}')
log(f'gate_sha256={gate_hash}')
log('PRECHECK_AND_GATE PASS')

In [ ]:
elan_home = RUN_ROOT / 'elan'
env = os.environ.copy()
env['ELAN_HOME'] = str(elan_home)
env['PATH'] = str(elan_home / 'bin') + os.pathsep + env['PATH']
installer = RUN_ROOT / 'elan-init.sh'
run(['curl', '--proto', '=https', '--tlsv1.2', '-sSfL',
    'https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh',
    '-o', str(installer)])
installer_hash = hashlib.sha256(installer.read_bytes()).hexdigest()
log(f'elan_installer_sha256={installer_hash}')
run(['sh', str(installer), '-y', '--no-modify-path', '--default-toolchain', 'none'], env=env)
run(['elan', 'toolchain', 'install', EXPECTED_TOOLCHAIN], env=env)
lean_version = run(['lean', '--version'], cwd=REPO, env=env).stdout
if 'version 4.29.0-rc6' not in lean_version or EXPECTED_LEAN_COMMIT not in lean_version:
    raise RuntimeError(f'Lean binary mismatch: {lean_version}')
run(['lake', '--version'], cwd=REPO, env=env)
run(['lake', 'exe', 'cache', 'get'], cwd=REPO, env=env)
log('TOOLCHAIN_AND_CACHE PASS')

In [ ]:
new_declarations = ['dualFieldBond', 'dualFieldScale', 'z2Bond_dual_factorization',
    'spatialKernel_dual_factorization', 'ringBondSum',
    'spatialWeightRing_eq_exp_ringBondSum', 'symWeighted_ring_dual_factorization']
module = run(['lake', 'build', 'YangMills.OS.SpatialDualBond'], cwd=REPO, env=env)
module_match = re.search(r'Build completed successfully \((\d+) jobs\)', module.stdout)
if not module_match or int(module_match.group(1)) != EXPECTED_MODULE_JOBS:
    raise RuntimeError(f'module job-count mismatch: {module_match.group(1) if module_match else "missing"}')
core = run(['lake', 'build', 'YangMillsCore'], cwd=REPO, env=env)
core_match = re.search(r'Build completed successfully \((\d+) jobs\)', core.stdout)
if not core_match or int(core_match.group(1)) != EXPECTED_CORE_JOBS:
    raise RuntimeError(f'core job-count mismatch: {core_match.group(1) if core_match else "missing"}')
oracle_run = run(['lake', 'env', 'lean', 'oracle_check.lean'], cwd=REPO, env=env)
(ARTIFACTS / 'oracle_output.txt').write_text(oracle_run.stdout, encoding='utf-8')
for name in new_declarations:
    marker = f"'YangMills.OS.{name}' depends on axioms:"
    if marker not in oracle_run.stdout:
        raise RuntimeError(f'permanent oracle omitted {name}')
allowed_axioms = {'propext', 'Classical.choice', 'Quot.sound'}
if 'sorryAx' in oracle_run.stdout:
    raise RuntimeError('oracle contains sorryAx')
for line in oracle_run.stdout.splitlines():
    if 'depends on axioms:' not in line:
        continue
    payload = line.split('depends on axioms:', 1)[1].strip().strip('[]')
    used = {item.strip() for item in payload.split(',') if item.strip()}
    if not used <= allowed_axioms:
        raise RuntimeError(f'nonstandard axioms: {used - allowed_axioms}')
run(['python3', 'scripts/check_consistency.py'], cwd=REPO, env=env)
metadata = {
  'repo_sha': actual_sha, 'toolchain': toolchain,
  'lean_commit': EXPECTED_LEAN_COMMIT, 'mathlib_pin': entries[0].get('rev'),
  'gate_sha256': gate_hash, 'module_jobs': int(module_match.group(1)),
  'core_jobs': int(core_match.group(1)),
  'utc_end': datetime.datetime.now(datetime.timezone.utc).isoformat(),
  'runtime': platform.platform(), 'cpu_count': os.cpu_count(),
  'memory': Path('/proc/meminfo').read_text(errors='replace').splitlines()[0],
  'elan_installer_sha256': installer_hash,
}
(ARTIFACTS / 'metadata.json').write_text(
    json.dumps(metadata, indent=2, sort_keys=True) + '\n', encoding='utf-8')
log(f'module_jobs_measured={module_match.group(1)}')
log(f'core_jobs_measured={core_match.group(1)}')
log('SPATIAL SYMWEIGHTED LEAN PASS')
hashes = {}
for path in sorted(ARTIFACTS.iterdir()):
    if path.name != 'SHA256SUMS':
        hashes[path.name] = hashlib.sha256(path.read_bytes()).hexdigest()
(ARTIFACTS / 'SHA256SUMS').write_text(
    ''.join(f'{digest}  {name}\n' for name, digest in hashes.items()), encoding='utf-8')
archive = shutil.make_archive('/content/spatial_symweighted_lean_artifacts', 'zip', ARTIFACTS)
log(f'artifact_zip={archive}')
log(f'artifact_zip_sha256={hashlib.sha256(Path(archive).read_bytes()).hexdigest()}')
from google.colab import files
files.download(archive)